In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel

In [2]:
load_dotenv(override=True)

True

In [6]:
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

In [7]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [8]:
#GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

In [9]:
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
#gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

deepseek_model = OpenAIChatCompletionsModel(model="minimax/minimax-m2:free", openai_client=openrouter_client)
gemini_model = OpenAIChatCompletionsModel(model="google/gemini-2.0-flash-exp:free", openai_client=openrouter_client)
gpt_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)

In [10]:
sales_agent1 = Agent(name="DeepSeek Sales Agent", instructions=instructions1, model=deepseek_model)
sales_agent2 =  Agent(name="Gemini Sales Agent", instructions=instructions2, model=gemini_model)
sales_agent3  = Agent(name="Llama3.3 Sales Agent",instructions=instructions3,model=llama3_3_model)

In [11]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [12]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("basitab208@gmail.com") 
    to_email = To("22dcs097@charusat.edu.in")  
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [48]:
# Multi-LLM health check — checks DeepSeek, Gemini, GROQ and OpenRouter/OpenAI if clients exist
# The cell is robust: it detects async vs sync client methods and reports success/error for each platform.
import asyncio
import inspect
from pprint import pprint

async def _check_one(name, client, model):
    if client is None:
        return {"name": name, "ok": False, "error": "client variable not found"}
    # Build a small ping prompt
    messages = [{"role": "user", "content": "Ping: please reply with a short OK and your model id."}]
    try:
        # Many AsyncOpenAI clients expose client.chat.completions.create as a coroutine function
        create_fn = None
        try:
            create_fn = client.chat.completions.create
        except Exception:
            # Some wrapper clients might expose chat_completions or different shape — attempt common alternatives
            create_fn = getattr(client, "chat_completion_create", None) or getattr(client, "chat_create", None) or None
        if create_fn is None:
            return {"name": name, "ok": False, "error": "no known chat completion method on client"}

        # Call the create function (may return a coroutine or a direct response)
        resp_candidate = create_fn(model=model, messages=messages, max_tokens=16, temperature=0)
        # Await only if it's awaitable/coroutine
        if inspect.isawaitable(resp_candidate):
            resp = await resp_candidate
        else:
            resp = resp_candidate

        # Try several ways to extract the assistant text from returned object/dict
        text = None
        try:
            # Newer SDK object-style response
            text = resp.choices[0].message.content
        except Exception:
            try:
                # dict-like responses
                text = resp['choices'][0]['message']['content']
            except Exception:
                # fallback to stringifying the response
                text = str(resp)

        return {"name": name, "ok": True, "reply": text}
    except Exception as e:
        return {"name": name, "ok": False, "error": repr(e)}

async def run_all_checks():
    # Candidate clients and typical model names used in this notebook
    candidates = [
        ("DeepSeek", globals().get('deepseek_client'), "minimax/minimax-m2:free"),
        ("Gemini", globals().get('gemini_client'), "gemini-2.0-flash"),
        ("GROQ", globals().get('groq_client'), "llama-3.3-70b-versatile"),
        ("GROQ", globals().get('groq_client'), "openai/gpt-oss-120b"),
    ]

    tasks = []
    missing = []
    for name, client, model in candidates:
        if client is None:
            missing.append({"name": name, "ok": False, "error": "client variable not present (skipped)"})
        else:
            tasks.append(_check_one(name, client, model))

    results = []
    if tasks:
        # run concurrently
        results = await asyncio.gather(*tasks)
    results.extend(missing)

    # Print a friendly report
    print('=== Multi-LLM Healthcheck Report ===')
    for r in results:
        if r.get('ok'):
            print(f"[OK]   {r['name']}: reply -> {str(r.get('reply'))[:150]}")
        else:
            print(f"[FAIL] {r['name']}: {r.get('error')}")
    print('=== End report ===')
    return results

# Execute checks from the notebook in a way that's compatible with Jupyter kernels
try:
    loop = asyncio.get_running_loop()
    if loop and loop.is_running():
        # We're inside a running event loop (typical for Jupyter). Use top-level await if supported.
        try:
            _health_results = await run_all_checks()
        except SyntaxError:
            # Some environments don't support top-level await in the executed cell; fall back to nest_asyncio
            import nest_asyncio
            nest_asyncio.apply()
            _health_results = asyncio.run(run_all_checks())
    else:
        _health_results = asyncio.run(run_all_checks())
except RuntimeError:
    # No running loop (standard Python REPL)
    _health_results = asyncio.run(run_all_checks())

pprint(_health_results)

=== Multi-LLM Healthcheck Report ===
[OK]   DeepSeek: reply -> 
[OK]   Gemini: reply -> OK. Gemini 1.5 Pro.

[OK]   GROQ: reply -> OK, LLaMA
[OK]   GROQ: reply -> 
=== End report ===
[{'name': 'DeepSeek', 'ok': True, 'reply': ''},
 {'name': 'Gemini', 'ok': True, 'reply': 'OK. Gemini 1.5 Pro.\n'},
 {'name': 'GROQ', 'ok': True, 'reply': 'OK, LLaMA'},
 {'name': 'GROQ', 'ok': True, 'reply': ''}]


In [13]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model=gpt_model)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model=gpt_model)
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [15]:
email_tools = [subject_tool, html_tool, send_html_email]

In [16]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=email_tools,
    model=gpt_model,
    handoff_description="Convert an email to HTML and send it")

In [17]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

In [18]:

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    model=gpt_model)

message = "Send out a cold sales email addressed to Dear CEO from Alice"


result = await Runner.run(sales_manager, message)

print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


**Subject:** Unlock 20% Faster Sales Growth with ComplAI’s AI‑Driven Sales Automation  

**Dear CEO,**

I’m Alice from ComplAI. Our AI‑powered sales automation platform is already helping dozens of high‑growth companies turn prospects into pipeline 20 % faster, while cutting manual effort by up to 70 %.

**What this means for you:**  
- **Automated lead scoring & qualification** – our models rank every lead in real time so your reps focus only on the highest‑intent prospects.  
- **Seamless CRM integration** – data syncs bi‑directionally with your existing Salesforce, HubSpot, or Zoho setup—no migration headaches.  
- **Personalized outreach at scale** – AI drafts hyper‑relevant emails, follow‑ups, and social touches, driven by each prospect’s behavior and firmographic profile.  
- **Actionable insights** – dashboards surface win‑rate drivers, conversion bottlenecks, and forecast confidence to keep your sales strategy data‑led.

A recent client, a mid‑market SaaS firm, reported a 28 % 

OPENAI_API_KEY is not set, skipping trace export


In [ ]:
from contextlib import contextmanager

@contextmanager
def trace(*args, **kwargs):
    # no-op tracer
    yield
